In [2]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [ ]:
# llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s up with you?'

In [ ]:
question = "I just discovered the course. Can I join now?"
# answer = llm(question)
# print(answer)

Yes—usually you can join after a course has started, but it depends on the course’s enrollment rules and whether there are still spots available.

If you want, I can help you draft a quick message to the instructor or registrar asking about late enrollment.


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
# answer = llm(prompt)
# answer

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [ ]:
# print(answer)

Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.


## Loading the Document

In [10]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [12]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [13]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [14]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [15]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [16]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [17]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [1]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [18]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [19]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [20]:
type(response)

openai.types.responses.response.Response

In [28]:
response.output[0].content[0].text

'Yes — you can still join now.\n\nIf you want to receive a certificate, you need to submit your project while submissions are still open.'

In [29]:
response.output_text

'Yes — you can still join now.\n\nIf you want to receive a certificate, you need to submit your project while submissions are still open.'

In [30]:
response.usage

ResponseUsage(input_tokens=334, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=32, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=366)

In [ ]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

response.output_text

In [ ]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

Compiling all functions into one cell

In [33]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

# Combining them all into one function: rag()

def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [34]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still being accepted.


# RAG Helper

## `ingest.py`

Let's put the ingestion process above into two functions
- `load_faq_data`: to get the documents
- `build_index`: to build search index

In [1]:
import requests
from minsearch import Index

In [2]:
def load_faq_data():
    docs_url = "https://datatalks.club/faq/json/courses.json"
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = "https://datatalks.club/faq"

    for course in courses_raw:
        course_url = f"""{url_prefix}{course["path"]}"""

        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()

        documents.extend(course_data)

    return documents

In [3]:
def build_index(documents):
    index = Index(
        text_fields=["question", "section", "answer"],
        keyword_fields=["course"]
    )

    index.fit(documents)
    return index

## `rag_helper.py`

Let's make our function more modular using a class, so we can change the search engine and AI Client API if we want, outside the RAG Helper

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

In [ ]:

def __init__(
    self,
    index,
    llm_client,
    instructions=INSTRUCTIONS,
    prompt_template=PROMPT_TEMPLATE,
    course="llm-zoomcamp",
    model="gpt-5.4-mini"
):
    self.index = index
    self.llm_client = llm_client
    self.instructions = instructions
    self.course = course
    self.prompt_template = prompt_template
    self.model = model

In [ ]:
def search(self, question, num_results=5):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": self.course}

    return self.index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=num_results
    )

def build_context(self, search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(self, question, search_results):
    context = self.build_context(search_results)
    prompt = self.prompt_template.format(
        question=question,
        context=context
    )
    return prompt #.strip()

def llm(self, user_prompt):
    message_history = [
        {"role": "developer", "content": self.instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = self.llm_client.responses.create(
        model=self.model,
        input=message_history
    )

    return response.output_text

# Combining them all into one function: rag()

def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt, model=self.model)
    return answer

# Using `ingest.py` & `rag_helper.py`

In [1]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want a certificate, you need to submit your project while submissions are still open.


# Persistent Search Engine: sqlitesearch

## Ingestion

In [2]:
from ingest import load_faq_data

documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Loaded 1208 documents


In [3]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 79 documents


In [4]:
import time
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

for doc in docs_llm:
    index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    time.sleep(0.5)

index.close()
print("Done. Index saved to faq.db")

Added: I just discovered the course. Can I still join?...
Added: Course: I have registered for the LLM Zoomcamp. When can I e...
Added: What is the video/zoom link to the stream for the “Office Ho...
Added: Cloud alternatives with GPU...
Added: Leaderboard: I am not on the leaderboard / how do I know whi...
Added: Certificate: Can I follow the course in a self-paced mode an...
Added: I missed the first homework - can I still get a certificate?...
Added: Homework: Why does the content keep changing?...
Added: When will the course be offered next?...
Added: Are there any lectures/videos? Where are they?...
Added: WSL2: ResponseError: model requires more system memory (X.X ...
Added: Server Error (500) When logging in to course homework using ...
Added: Why are we not using Langchain in the course?...
Added: OpenAI: Error when running OpenAI chat.completions.create co...
Added: OpenAI: Error: RateLimitError: Error code: 429 -...
Added: OpenAI: Error: 'Cannot import name OpenAI from openai

## Get the Documents

In [5]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [6]:
sqlite_index.count()

79

In [7]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'Do we submit 2 projects, what does attempt 1 and 2 mean?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'I am using Azure OpenAI and I am still getting an error of Error code: 400 - {\'error\': {\'message\': "Missing required parameter: \'tools[0].function\'.", \'type\': \'invalid_request_error\', \'param\': \'tools[0].function\', \'code\': \'missing_required_parameter\'}}?']

## RAG with sqlitesearch

In [8]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [9]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join after it started. If you want a certificate, you need to submit your project while the course is still accepting submissions.


## Closing DB Connection

In [10]:
sqlite_index.close()

# RAG with Agents

Instead of _us_ doing the search, we want the _LLM to do the search_ by itself, and decide when to do it. To achieve this, we need to define the `search` function, and make a description about it in JSON format. This way, the LLM would read the JSON, recognizes our search function, and utilize it as a **tool** to help itself answer questions.

In [2]:
from sqlitesearch import TextSearchIndex

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
    )

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [3]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [4]:
from openai import OpenAI

openai_client = OpenAI()

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join the course discovered course can I join late enrollment enrollment after course starts"}', call_id='call_d1M5lDdFdpzCRpnf6OcqNNe3', name='search', type='function_call', id='fc_07e479a5f7396a11006a379adc8b60819b8d4219a5baef9131', namespace=None, status='completed')]

In [5]:
import json

call = response.output[0]
args = json.loads(call.arguments)

# args

results = search(**args)
result_json = json.dumps(results, indent=2)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [6]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [7]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join the course discovered course can I join late enrollment enrollment after course starts"}', call_id='call_d1M5lDdFdpzCRpnf6OcqNNe3', name='search', type='function_call', id='fc_07e479a5f7396a11006a379adc8b60819b8d4219a5baef9131', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_d1M5lDdFdpzCRpnf6OcqNNe3',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-p

Get another response from the LLM based on the addedd messages

In [8]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [9]:
response.output

[ResponseOutputMessage(id='msg_07e479a5f7396a11006a379bd4f044819b8c46b9703351a2ac', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [10]:
response.usage

ResponseUsage(input_tokens=692, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=31, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=723)

## Agentic Loop

What if searching for 1 time doesn't bring up the desired answer, probably because we are asking the LLM 2 different things in a single question? We can make the LLM decide whether or not it is required to search again, and if it does, it can also do the search, multiple time

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [12]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [15]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    print(item)
    
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join registration FAQ"}', call_id='call_X3cXIOgUWRE5UQIBBOUA7Jss', name='search', type='function_call', id='fc_0740dda0107d6e4b006a37a21f0ba0819aa39ae16f5b41fde3', namespace=None, status='completed')
function_call: search {"query":"join course discovered course can I join enrollment late join registration FAQ"}


In [16]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        print(item)
        
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ResponseOutputMessage(id='msg_0740dda0107d6e4b006a37a2205af4819a902d0a4d831e3368', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.\n\nIf you’re just following along, you can start learning right away; registration isn’t required to begin. Would you like me to help with how to get started or explain the certificate requirements?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open.

If you’re just following along, you can start learning right away; registration isn’t required to begin. Would you like me to help with how to get started or explain the certificate requirements?


## Wrapping it in a Function

In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            print(item)
            
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [18]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run install Ollama locally run model start server command"}
iteration #2...
function_call: search {"query":"Ollama local server localhost 11434 ollama run llama3 python client pip install ollama"}
iteration #3...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - Download it from: https://ollama.com/download
   - **macOS**: install the `.pkg`
   - **Windows**: install the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model, starts it locally, and opens a chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing the Ollama server is up.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   Then:
   ```python
   import ollama

   response = ollama.chat(
       model='lla

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - Download it from: https://ollama.com/download\n   - **macOS**: install the `.pkg`\n   - **Windows**: install the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model, starts it locally, and opens a chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response showing the Ollama server is up.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   Then:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you want, I can also show you how to run Ollama with a different model or how to connect it to your Python app.'

Let's change the instructions a little bit to encourage the LLM to perform more searches and prevent it from answering off-topic questions

In [20]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [21]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit what is queen gambit course FAQ"}
iteration #2...
function_call: search {"query":"queen gambit chess queen's gambit FAQ"}
iteration #3...
ASSISTANT:
I’m not finding anything in the course FAQ that matches “queen gambit,” so this looks like it may be off-topic or misspelled.

If you meant something course-related, try rephrasing with more context and I can look again. Otherwise, is there another area of the course you want to explore?


'I’m not finding anything in the course FAQ that matches “queen gambit,” so this looks like it may be off-topic or misspelled.\n\nIf you meant something course-related, try rephrasing with more context and I can look again. Otherwise, is there another area of the course you want to explore?'

## Using ToyAIKit Framework

In [22]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

We can ask ToyAIKit to build a `search_tool` JSON for us, but in order to do that, we need to add a docstring inside our `search` function

In [38]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
    )

    search_result = index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

    index.close()
    return search_result

In [39]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [40]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

## Chat interface and runner

In [41]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [42]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


To answer the LLM response and continue the conversation, we need to add `previous_messages`

In [35]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


For a chat-like workflow, run the built-in input loop:

In [36]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='hello', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"hello"}', call_id='call_6VfqkdO6LfZ55dgzai7Pi7sz', name='searc

In [37]:
index

NameError: name 'index' is not defined